This notebook implements all Machine Learning, non-deep regressors (Fig 2B; SFig 3). 

Specifically, we provide code for:
```
- training Ridge, Support Vector, Multilayer Perceptron and 
  Random Forest regressors for individual mutational series
- multiple encoding options for the input DNA sequences
- multiple training size options
```


## 0. Mount Drive and set the main path

In [ ]:
#mount to drive
from google.colab import drive
try:
  drive.mount('/content/drive', force_remount=True)
except:
  drive._mount('/content/drive', force_remount=True)

# construct the path to use
import os
 
# get the current working directory
dir_path = os.path.dirname(os.path.realpath('__file__'))

# loop to crawl over your Drive and construct the correct path
for root, dirs, files in os.walk(dir_path):
    for file in files:
 
        # do not change the extension *.ipynb*
        if file.endswith('1_kmers.ipynb'):
          # check that we're using the correct parent directory
          if f'{root}/{file}'[-24:] == '/seq2yield/1_kmers.ipynb':
            # create path to the import directory
            drive_dir = f"{f'{root}/{file}'[:-13]}to_import/"

drive_dir

## 1. Import libraries and get data 

In [ ]:
import re
import random
from random import randint
import scipy
import scipy.stats
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

import sys
sys.path.insert(0, drive_dir)

import utils

# LOAD .csv into a DataFrame
pd.options.display.max_rows = 10
raw_data = pd.read_csv(f'{drive_dir}Ecoli_data.csv', index_col= 0).dropna().reset_index(drop = True)

## 2. Generate or load sets.
```
working_set: genotype-phenotype pairs to split for training in various percentages
heldout_set: genotype-phenotype pairs to use **only** for testing
```

In [ ]:
iter_ID   = 1
get_saved = True #set to *True* to use saved splits

###
iteration = f'iteration_{iter_ID}/'
series_dict_raw = utils.split_seeds(raw_data)

working_set, heldout_set = utils.get_split(series_dict = series_dict_raw, 
                                           series_list = [sr for sr in range(1,57)],
                                           load_saved = get_saved,
                                           train_size = 0.9, #change to get different dataset ratios
                                           path = f'{drive_dir}_saved/{iteration}')

print(f'Working set contains {working_set.shape[0]} sequences train and validation\nHeldout contains {heldout_set.shape[0]} sequences for testing')

## 3. Define hyperparameters for *non-deep* regressors

In [ ]:
#----
print_res = False

# regressors' default params
n_estimator = 25       
max_depth = 30           
min_samples_leaf = 3    
min_samples_split = 2   

rf_params = [n_estimator, 
             max_depth, 
             min_samples_leaf, 
             min_samples_split,
             print_res]

#----
hidden_layer_number = 3      
hidden_layer_size = 100       
max_iter = 225         

mlp_params = [hidden_layer_number, 
              hidden_layer_size, 
              max_iter,
              print_res]

#----
kernel = 'rbf'   
C = 30          
epsilon = 0.5    

svr_params = [kernel,
              C,
              epsilon,
              print_res]

#----
alphalist = 10**(np.linspace(-1,2,50))
cv = 7   

ridge_params = [alphalist, 
                cv, 
                print_res]

## 4. Non-deep regressors

Code below creates combinations of training size, encodings, and regressors for individual mutational series. 

**Note**: running the code as is, will attempt to train *56 × 6 × 12 × 4 = 16,128 models* and crash the virtual machine's RAM. For this reason, I suggest executing the block in separate runs.

### 4.1 Set options for training
```
stratify (->bool): *True* for stratified; *False* for non-stratified
series_lst (->list): list of mutational series (type:int)
train_sz_lst (->list): list of training sizes (type:float)
encodings (->list): list of encodings for DNA sequences (type:str)
```

In [ ]:
stratify = True #set to *False* to do non-stratified splits
pheno = 'protein' #do not change this option -- protein expression is the only target variable provided
series_lst = [sd for sd in range(1, 57)] #change (lower, upper) bounds for *range* to execute different combinations of mutational series
train_sz_lst = [0.05, 0.10, 0.25, 0.50, 0.75]
encodings = ['one-hot', \
             '1-mers', \
             '3-mers', '4-mers', '5-mers', '6-mers',\
             '3-count', '4-count','5-count','6-count',\
             'features', 'mixed'
             ]

### 4.2 Run the pipeline

In [ ]:
from models_misc import custom_ridge, custom_SVR, custom_MLP, custom_rf

series_dict = utils.split_seeds(working_set)
test_series_dict = utils.split_seeds(heldout_set)

# organize hyperparameters
reg_params = dict({'rf' : rf_params,
                   'mlp' : mlp_params,
                   'svr' : svr_params,
                   'ridge' : ridge_params
                   })

#----
results = None
for no_seed in series_lst:
  # create dataframes for each series
  sd_res = None
  test_series_df = test_series_dict[f'seed_{no_seed}'].reset_index(drop=True)

  # start train size loop
  for train_sz in train_sz_lst:
    train_df, non_train_df = utils.get_split(series_dict = series_dict, 
                                             series_list = [no_seed],
                                             train_size  = train_sz
                                             )

    # create syn_dna() objects
    emb_dna = utils.syn_dna(train_df.reset_index(drop=True))
    test_emb_dna = utils.syn_dna(test_series_df.reset_index(drop=True))

    #lists to store predictions
    rf_lst, mlp_lst, svr_lst, ridge_lst = [], [], [], []

    # start encodings loop
    for embd_method in encodings:
      if embd_method == 'features':
        # initialize the scaler object
        scaler = MinMaxScaler()
        # fit the scaler on the train dataframe to avoid information leakage from the test set
        scaler.fit(emb_dna.feat_vector.values)
        # transform biophysical properties vectors
        embd = scaler.transform(emb_dna.feat_vector.values)
        test_embd = scaler.transform(test_emb_dna.feat_vector.values)
      else:
        embd = utils.get_encoding(emb_dna, embd_method=embd_method)
        test_embd = utils.get_encoding(test_emb_dna, embd_method=embd_method)
      
      # define target distribution
      phenotype = emb_dna.protein
      phenotype_test = test_emb_dna.protein

      # organize in a dictionary
      sets = dict({'train' : [embd, phenotype],
                   'test'  : [test_embd, phenotype_test]})
      
      # start regressor loop
      for reg in reg_params.keys():
        # random forest
        if reg == 'rf':
          rf, R2 = custom_rf(*reg_params[reg], **sets)
          rf_lst.append(R2)
        # multilayer perceptron
        elif reg == 'mlp':
          mlp, R2 = custom_MLP(*reg_params[reg], **sets)
          mlp_lst.append(R2)   
        # support vector regressor
        elif reg == 'svr':
          svr, R2 = custom_SVR(*reg_params[reg], **sets)
          svr_lst.append(R2)   
        # ridge 
        elif reg == 'ridge':
          ridge, R2 = custom_ridge(*reg_params[reg], **sets)
          ridge_lst.append(R2)

    # organize results per series
    try:
      if sd_res == None:
        sd_res = pd.DataFrame({'seed' : [no_seed for itr in range(len(encodings))],
                               'encoding': [enc for enc in encodings],
                               f'ridge_{train_sz}' : ridge_lst,
                               f'MLP_{train_sz}' : mlp_lst,                               
                               f'SVR_{train_sz}' : svr_lst,
                               f'RF_{train_sz}' : rf_lst,
                               })
    except:
      if sd_res.empty == False:
        temp_df = pd.DataFrame({f'ridge_{train_sz}' : ridge_lst,
                                f'MLP_{train_sz}' : mlp_lst,                                
                                f'SVR_{train_sz}' : svr_lst,
                                f'RF_{train_sz}' : rf_lst,
                                })
      sd_res = pd.concat([sd_res, temp_df], axis=1) #concatenate data split df's horizontally

  # organize all results in single dataframe
  try:
    if results == None:
      results = sd_res.copy()
  except:
    if results.empty == False:
      results = pd.concat([results, sd_res], ignore_index=True, axis=0) #concatenate series results df's vertically

display(results)